In [ ]:
import requests
import pandas as pd

# Function 1: Convert city name to latitude/longitude
def get_coordinates(city):
    url = "https://geocoding-api.open-meteo.com/v1/search"
    params = {"name": city, "count": 1}

    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        data = response.json()

        if "results" not in data:
            print(f"City '{city}' not found.")
            return None, None

        lat = data["results"][0]["latitude"]
        lon = data["results"][0]["longitude"]
        return lat, lon

    except requests.exceptions.RequestException as e:
        print("Error fetching location data:", e)
        return None, None


# Function 2: Get hourly forecast using latitude/longitude
def get_weather(lat, lon):
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat,
        "longitude": lon,
        "hourly": "temperature_2m,rain,windspeed_10m",
        "timezone": "auto"
    }

    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        data = response.json()
        return data

    except requests.exceptions.RequestException as e:
        print("Error fetching weather data:", e)
        return None


# Function 3: Save forecast data to CSV
def save_to_csv(city, data, hours=8):
    time = data["hourly"]["time"][:hours]
    temperature = data["hourly"]["temperature_2m"][:hours]
    rain = data["hourly"]["rain"][:hours]
    windspeed = data["hourly"]["windspeed_10m"][:hours]

    df = pd.DataFrame({
        "Time": time,
        "Temperature (°C)": temperature,
        "Rain (mm)": rain,
        "Windspeed (km/h)": windspeed
    })

    filename = f"{city}_weather.csv"
    df.to_csv(filename, index=False)
    print(f"\nForecast saved to '{filename}'")
    print(df)


# Main Function
def main():
    while True:
        city = input("Enter city name: ")

        lat, lon = get_coordinates(city)

        if lat is not None and lon is not None:
            data = get_weather(lat, lon)
            if data is not None:
                save_to_csv(city, data)

        choice = input("\nCheck another city? (yes/no): ").lower()
        if choice != "yes":
            print("Program ended.")
            break


# Function call
if __name__ == "__main__":
    main()

Enter city name: Indore

Forecast saved to 'Indore_weather.csv'
               Time  Temperature (°C)  Rain (mm)  Windspeed (km/h)
0  2026-10-22T00:00              23.5        0.0               5.5
1  2026-10-22T01:00              23.3        0.0               7.4
2  2026-10-22T02:00              22.9        0.0               5.6
3  2026-10-22T03:00              22.7        0.0               5.3
4  2026-10-22T04:00              22.4        0.0               6.4
5  2026-10-22T05:00              22.2        0.0               6.0
6  2026-10-22T06:00              22.2        0.0               6.8
7  2026-10-22T07:00              23.3        0.0               7.2

Check another city? (yes/no): YES
Enter city name: bhopal

Forecast saved to 'bhopal_weather.csv'
               Time  Temperature (°C)  Rain (mm)  Windspeed (km/h)
0  2026-10-22T00:00              23.3        0.0               1.4
1  2026-10-22T01:00              23.0        0.0               1.4
2  2026-10-22T02:00              